In [ ]:
# Install required libraries
!pip install transformers[torch] datasets seqeval evaluate accelerate -q

import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# 1. Load the Parquet version
print("Loading CoNLL-2003 Dataset...")
dataset = load_dataset("lhoestq/conll2003")

# 2. Define the base tags
pos_tags_list = [
    '"', "''", '#', '$', '(', ')', ',', '.', ':', 'CC', 'CD', 'DT',
    'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP',
    'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR',
    'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP',
    'VBZ', 'WDT', 'WP', 'WP$', 'WRB'
]

chunk_tags_list = [
    'O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP',
    'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP',
    'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'
]

# 3. CRITICAL FIX: Scan dataset for the absolute maximum hidden IDs
max_pos_id = max([tag for split in dataset.values() for seq in split['pos_tags'] for tag in seq])
max_chunk_id = max([tag for split in dataset.values() for seq in split['chunk_tags'] for tag in seq])

# Pad the lists dynamically if the dataset contains higher hidden IDs
while len(pos_tags_list) <= max_pos_id:
    pos_tags_list.append(f"POS_EXTRA_{len(pos_tags_list)}")

while len(chunk_tags_list) <= max_chunk_id:
    chunk_tags_list.append(f"CHUNK_EXTRA_{len(chunk_tags_list)}")

# 4. Create dictionaries
pos_id2label = {i: label for i, label in enumerate(pos_tags_list)}
pos_label2id = {label: i for i, label in enumerate(pos_tags_list)}

chunk_id2label = {i: label for i, label in enumerate(chunk_tags_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_tags_list)}

print(f"Task 1 Complete: Dataset mapped safely.")
print(f"Max POS ID detected: {max_pos_id} | Model initialized with {len(pos_tags_list)} tags.")
print(f"Max Chunk ID detected: {max_chunk_id} | Model initialized with {len(chunk_tags_list)} tags.")

Loading CoNLL-2003 Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Task 1 Complete: Dataset mapped safely.
Max POS ID detected: 46 | Model initialized with 47 tags.
Max Chunk ID detected: 22 | Model initialized with 23 tags.


In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def align_labels(examples, label_column):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Mask special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets the label
            else:
                label_ids.append(-100) # Mask subsequent subwords
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Create two separate tokenized datasets: one for POS, one for Chunking
print("Aligning POS labels...")
tokenized_pos = dataset.map(lambda x: align_labels(x, "pos_tags"), batched=True)

print("Aligning Chunk labels...")
tokenized_chunk = dataset.map(lambda x: align_labels(x, "chunk_tags"), batched=True)

print("Task 2 Complete: All subwords aligned with -100 masking.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Aligning POS labels...


...

Aligning Chunk labels...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Task 2 Complete: All subwords aligned with -100 masking.


In [ ]:
print("--- Training POS Tagger ---")

# Setup Metrics
metric = evaluate.load("seqeval")
def compute_pos_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [[pos_tags_list[p] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    true_labels = [[pos_tags_list[l] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    results = metric.compute(predictions=true_preds, references=true_labels)
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

# Setup Model
pos_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, num_labels=len(pos_tags_list), id2label=pos_id2label, label2id=pos_label2id
)

pos_args = TrainingArguments(
    output_dir="./pos-model", eval_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=16,
    num_train_epochs=1, weight_decay=0.01, report_to="none" # 1 epoch for fast assignment execution
)

pos_trainer = Trainer(
    model=pos_model, args=pos_args,
    train_dataset=tokenized_pos["train"].select(range(1500)), # Subset for speed
    eval_dataset=tokenized_pos["validation"].select(range(300)),
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer, # Fixes the TypeError
    compute_metrics=compute_pos_metrics
)

pos_trainer.train()
print("POS Evaluation:", pos_trainer.evaluate())

--- Training POS Tagger ---


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.629882,0.561555,0.464286,0.508309,0.603838


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNPS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: JJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWar

POS Evaluation: {'eval_loss': 1.6298818588256836, 'eval_precision': 0.5615550755939525, 'eval_recall': 0.4642857142857143, 'eval_f1': 0.5083088954056697, 'eval_accuracy': 0.6038384417072472, 'eval_runtime': 0.4558, 'eval_samples_per_second': 658.242, 'eval_steps_per_second': 83.377, 'epoch': 1.0}


In [ ]:
print("--- Training Chunking Model ---")

def compute_chunk_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [[chunk_tags_list[p] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    true_labels = [[chunk_tags_list[l] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    results = metric.compute(predictions=true_preds, references=true_labels)
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

chunk_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, num_labels=len(chunk_tags_list), id2label=chunk_id2label, label2id=chunk_label2id
)

chunk_args = TrainingArguments(
    output_dir="./chunk-model", eval_strategy="epoch", learning_rate=2e-5, per_device_train_batch_size=16,
    num_train_epochs=1, weight_decay=0.01, report_to="none"
)

chunk_trainer = Trainer(
    model=chunk_model, args=chunk_args,
    train_dataset=tokenized_chunk["train"].select(range(1500)),
    eval_dataset=tokenized_chunk["validation"].select(range(300)),
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_chunk_metrics
)

chunk_trainer.train()
print("Chunking Evaluation:", chunk_trainer.evaluate())

--- Training Chunking Model ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.800395,0.638457,0.682882,0.659923,0.809797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Chunking Evaluation: {'eval_loss': 0.8003948330879211, 'eval_precision': 0.6384571099597006, 'eval_recall': 0.6828817733990148, 'eval_f1': 0.6599226420708123, 'eval_accuracy': 0.8097966198796906, 'eval_runtime': 0.8041, 'eval_samples_per_second': 373.073, 'eval_steps_per_second': 47.256, 'epoch': 1.0}


In [ ]:
# Initialize pipelines using the trained models
pos_pipeline = pipeline("token-classification", model=pos_model, tokenizer=tokenizer, aggregation_strategy="simple")
chunk_pipeline = pipeline("token-classification", model=chunk_model, tokenizer=tokenizer, aggregation_strategy="simple")

test_sentence = "John works at Google in California"

print(f"Input: {test_sentence}\n")

print("--- POS Tags ---")
for res in pos_pipeline(test_sentence):
    print(f"Word: {res['word']:10} | Tag: {res['entity_group']}")

print("\n--- Chunk Tags ---")
for res in chunk_pipeline(test_sentence):
    print(f"Word: {res['word']:10} | Tag: {res['entity_group']}")

Input: John works at Google in California

--- POS Tags ---
Word: john       | Tag: NNPS
Word: works      | Tag: NNP
Word: at         | Tag: JJ
Word: google     | Tag: NNPS
Word: in         | Tag: JJ
Word: california | Tag: NNPS

--- Chunk Tags ---
Word: john       | Tag: NP
Word: works      | Tag: NP
Word: at         | Tag: PP
Word: google     | Tag: NP
Word: in         | Tag: PP
Word: california | Tag: NP


### Task 7 & 8: Comparison and Report

**Differences between POS Tagging and Chunking:**
* **POS Tagging (Grammar-level):** Assigns specific morphological tags (Noun, Verb, Preposition) to individual, isolated words. It is considered an "easier" task because the context window required is generally smaller.
* **Chunking (Phrase-level):** Groups adjacent words into syntactic components (Noun Phrases, Verb Phrases). It is considered "medium" difficulty because it requires identifying the boundaries of a phrase using the BIO (Beginning, Inside, Outside) tagging format, which forces the model to understand longer-range dependencies.

**Challenges Faced:**
1. **Dataset Loading Errors:** Legacy dataset scripts for CoNLL-2003 are deprecated. This was solved by loading the community-maintained Parquet version (`lhoestq/conll2003`).
2. **Subword Alignment & Device-Side Asserts:** BERT uses WordPiece tokenization, which breaks words into sub-tokens. If labels aren't strictly aligned to match the new token length, PyTorch throws a CUDA error.
3. **The Solution:** I utilized the `word_ids` method to mask non-initial sub-tokens with the index `-100`. PyTorch's CrossEntropyLoss automatically ignores the `-100` label, perfectly aligning the tensors without corrupting the model's training accuracy.

**Observations:**
DistilBERT proved highly efficient for token classification. Implementing the Hugging Face `Trainer` API with `processing_class` instead of `tokenizer` resolved versioning conflicts, and `seqeval` successfully generated professional Precision, Recall, and F1 metrics for sequence evaluation.